# Retry Classifier Dataset EDA

Use this notebook after collecting `features.csv` with `retry_capture_features`.

Expected deployable inputs:

- `center_image_path`
- `pred_xy_offset_mm`
- `fz`
- `delta_fz`
- `fxy_norm`
- `cmd_insert_depth_mm`

`success_event_observed` and `/scoring`-derived values are label-only and must not be used as model inputs.

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from PIL import Image
except ImportError:
    Image = None

DATASET_DIR = Path(os.environ.get("AIC_RETRY_DATASET_DIR", "/tmp/retry_classifier_dataset")).expanduser()
CSV_PATH = Path(os.environ.get("AIC_RETRY_FEATURE_CSV", str(DATASET_DIR / "features.csv"))).expanduser()

CORE_FEATURES = [
    "pred_xy_offset_mm",
    "fz",
    "delta_fz",
    "fxy_norm",
    "cmd_insert_depth_mm",
]
LABEL_COLUMNS = ["class_name", "binary_success", "label_reason", "intended_class"]
IMAGE_COL = "center_image_path"

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

CSV_PATH

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"rows={len(df)} cols={len(df.columns)}")
display(df.head())

missing = [c for c in [IMAGE_COL, *CORE_FEATURES, "binary_success", "class_name"] if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

In [ ]:
for col in CORE_FEATURES + ["binary_success"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (100.0 * df.isna().mean()).round(2),
})
display(summary.loc[[c for c in [*LABEL_COLUMNS, IMAGE_COL, *CORE_FEATURES, "success_event_observed", "sample_count"] if c in summary.index]])

print("binary_success counts")
display(df["binary_success"].value_counts(dropna=False).rename_axis("binary_success").to_frame("count"))

print("class counts")
display(df["class_name"].value_counts(dropna=False).rename_axis("class_name").to_frame("count"))

In [ ]:
display(df[CORE_FEATURES].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

group_cols = ["class_name", "binary_success"]
display(
    df.groupby(group_cols, dropna=False)[CORE_FEATURES]
      .agg(["count", "mean", "std", "min", "median", "max"])
      .round(3)
)

In [ ]:
fig, axes = plt.subplots(len(CORE_FEATURES), 1, figsize=(9, 3 * len(CORE_FEATURES)))
for ax, feature in zip(axes, CORE_FEATURES):
    for label, color in [(0, "tab:red"), (1, "tab:blue")]:
        values = df.loc[df["binary_success"] == label, feature].dropna()
        ax.hist(values, bins=30, alpha=0.45, label=f"success={label}", color=color)
    ax.set_title(feature)
    ax.legend()
plt.tight_layout()

In [ ]:
def scatter_by_class(x, y):
    fig, ax = plt.subplots(figsize=(7, 5))
    for class_name, group in df.groupby("class_name", dropna=False):
        ax.scatter(group[x], group[y], s=28, alpha=0.75, label=str(class_name))
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.legend(loc="best", fontsize=8)
    ax.set_title(f"{x} vs {y}")
    plt.show()

scatter_by_class("pred_xy_offset_mm", "delta_fz")
scatter_by_class("pred_xy_offset_mm", "fxy_norm")
scatter_by_class("cmd_insert_depth_mm", "delta_fz")

In [ ]:
corr = df[CORE_FEATURES + ["binary_success"]].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.index)), corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax)
ax.set_title("Feature correlation")
plt.tight_layout()

In [ ]:
def resolve_image_path(value):
    if pd.isna(value) or str(value).strip() == "":
        return None
    path = Path(str(value)).expanduser()
    if not path.is_absolute():
        path = CSV_PATH.parent / path
    return path if path.exists() else None

def show_image_grid(frame, n_per_class=4, seed=0):
    if Image is None:
        print("PIL is not installed; image preview skipped.")
        return
    classes = list(frame["class_name"].dropna().unique())
    if not classes:
        print("No class_name values available.")
        return
    fig, axes = plt.subplots(len(classes), n_per_class, figsize=(3 * n_per_class, 3 * len(classes)))
    axes = np.asarray(axes).reshape(len(classes), n_per_class)
    rng = np.random.default_rng(seed)
    for row_idx, class_name in enumerate(classes):
        group = frame[frame["class_name"] == class_name]
        if len(group) > n_per_class:
            group = group.sample(n_per_class, random_state=seed)
        for col_idx in range(n_per_class):
            ax = axes[row_idx, col_idx]
            ax.axis("off")
            if col_idx >= len(group):
                continue
            sample = group.iloc[col_idx]
            path = resolve_image_path(sample.get(IMAGE_COL, ""))
            if path is None:
                ax.set_title("missing image", fontsize=8)
                continue
            img = Image.open(path)
            ax.imshow(img)
            ax.set_title(f"{class_name}\nxy={sample['pred_xy_offset_mm']:.1f}, dFz={sample['delta_fz']:.1f}", fontsize=8)
    plt.tight_layout()

show_image_grid(df, n_per_class=4, seed=7)

In [ ]:
leakage_cols = [
    "success_event_observed",
    "latest_insertion_event",
    "label_reason",
    "class_name",
    "binary_success",
]
available_leakage_cols = [c for c in leakage_cols if c in df.columns]
print("Do not use these as model inputs:")
print(available_leakage_cols)

clean = df[[c for c in ["episode_id", IMAGE_COL, *CORE_FEATURES, "binary_success", "class_name"] if c in df.columns]].copy()
display(clean.head())

clean_path = CSV_PATH.parent / "features_model_inputs_preview.csv"
clean.to_csv(clean_path, index=False)
print(f"wrote {clean_path}")